## Libraries import

In [3]:
pip install clickhouse_connect

Note: you may need to restart the kernel to use updated packages.


In [25]:
import clickhouse_connect
import pandas as pd

## Apis and clickhouse connection

In [22]:
API_KEY = "67ad1bea200726.95055451"
BASE_URL = "https://eodhd.com/api"

In [23]:
CLICKHOUSE_HOST = '54.234.38.203'
CLICKHOUSE_PORT = 8123
CLICKHOUSE_USER = 'chain8'
CLICKHOUSE_PASS = 'c8_2025'
CLICKHOUSE_DB = 'default'

In [8]:
client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASS,
    database=CLICKHOUSE_DB
)

In [9]:
# ✅ Fetch symbol list from metadata
df_tsx = client.query_df("""
    SELECT *
    FROM tsx_eod
""")

In [10]:
df_tsx.head()

,code,date,open,high,low,close,adjusted_close,volume,created_on
0,AAB,2025-06-26,0.035,0.04,0.035,0.04,0.04,23700,2025-06-27 15:11:58
1,AAPL,2025-06-26,29.190,29.33,28.890,29.17,29.17,230686,2025-06-27 15:11:58
2,AAUC,2025-06-26,18.500,18.60,18.260,18.59,18.59,114134,2025-06-27 15:11:58
3,AAV,2025-06-26,11.530,11.64,11.450,11.60,11.60,381999,2025-06-27 15:11:58
4,ABRA,2025-06-26,4.620,4.95,4.600,4.92,4.92,224226,2025-06-27 15:11:58


## eda

In [11]:
df_tsx.shape

(280448, 9)

In [27]:
df_tsx.head()

,code,date,open,high,low,close,adjusted_close,volume
0,AAB,2025-06-26,0.035,0.04,0.035,0.04,0.04,23700
1,AAPL,2025-06-26,29.190,29.33,28.890,29.17,29.17,230686
2,AAUC,2025-06-26,18.500,18.60,18.260,18.59,18.59,114134
3,AAV,2025-06-26,11.530,11.64,11.450,11.60,11.60,381999
4,ABRA,2025-06-26,4.620,4.95,4.600,4.92,4.92,224226


In [28]:
# Ensure 'date' is datetime
df_tsx['date'] = pd.to_datetime(df_tsx['date'])

# Get the last available date in the data
last_date = df_tsx['date'].max()

# Filter last 30 trading days
start_date = last_date - pd.Timedelta(days=60)  # buffer for weekends
last_30_trading_days = df_tsx[df_tsx['date'] >= start_date].copy()

# Calculate daily return (high - open)
last_30_trading_days['daily_return'] = last_30_trading_days['high'] - last_30_trading_days['open']

# Calculate average daily return for each stock
avg_returns = last_30_trading_days.groupby('code')['daily_return'].mean().reset_index()

# Rename column
avg_returns.rename(columns={'daily_return': 'avg_max_return'}, inplace=True)

# Rank the stocks based on avg_max_return (highest return = rank 1)
avg_returns['rank'] = avg_returns['avg_max_return'].rank(ascending=False, method='dense').astype(int)

# Sort by rank
avg_returns = avg_returns.sort_values(by='rank')

print(avg_returns)


      code  avg_max_return  rank
181    CSU       51.350707     1
280    FFH       21.508833     2
249    ELF        6.363902     3
154    CLS        4.174186     4
369    IFC        2.780698     5
..     ...             ...   ...
729  WCM-A        0.000000   744
750    WRX        0.000000   744
606    SBR        0.000000   744
656   SZLS        0.000000   744
497   NUMI        0.000000   744

[766 rows x 3 columns]


In [29]:
avg_returns.head()

,code,avg_max_return,rank
181,CSU,51.350707,1
280,FFH,21.508833,2
249,ELF,6.363902,3
154,CLS,4.174186,4
369,IFC,2.780698,5


In [31]:
df_tsx[df_tsx['code']=='CSU']

,code,date,open,high,low,close,adjusted_close,volume
180,CSU,2025-06-26,4909.1299,4946.7002,4844.8799,4926.7900,4926.7900,21575
4739,CSU,2025-05-27,4850.1001,4930.6099,4837.1201,4870.1299,4868.7585,42300
4740,CSU,2025-05-28,4852.3101,4940.0000,4852.3101,4869.4199,4868.0487,26900
4741,CSU,2025-05-29,4872.1401,4925.9399,4817.2598,4896.5801,4895.2012,22900
4742,CSU,2025-05-30,4894.6099,4985.7202,4861.7598,4975.7598,4974.3586,99900
...,...,...,...,...,...,...,...,...
79217,CSU,2025-05-20,5020.0200,5124.8999,5012.6099,5050.4800,5049.0578,34700
79218,CSU,2025-05-21,5063.5601,5063.6602,4862.1802,4880.7100,4879.3356,39600
79219,CSU,2025-05-22,4912.2300,4930.7700,4860.0200,4900.0000,4898.6201,29400
79220,CSU,2025-05-23,4900.0000,5030.2998,4832.7700,4841.0698,4839.7065,30400
